<a href="https://colab.research.google.com/github/usama488/bioinformatics-analysis/blob/main/RNASeq_Differential_Expression_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  RNA-Seq Differential Expression Analysis



# Is notebook mein hum RNA-Seq gene expression data ka Differential Expression (DE) Analysis karain gay  yani do groups (jaise **Tumor vs Normal**) ke darmiyan konsy genes significantly zyada ya kam expressed hain, yeh nikalain gay.

**Pipeline:**
1. Data load / simulate (real count matrix se replace kar sakte hain)
2. Normalization (CPM + log2 transform)
3. Differential Expression testing (t-test + Benjamini-Hochberg correction)
4. Interactive Plotly visualizations — PCA, Volcano plot, Heatmap, Boxplots
5. Top DE genes se ek classifier (Tumor vs Normal predictor) train karna
6. Runtime cell — apna khud ka sample expression data daal kar direct predict karein




## 1. Setup & Imports

In [10]:
# !pip install -q plotly scikit-learn pandas numpy statsmodels ipywidgets

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy import stats
from statsmodels.stats.multitest import multipletests

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

np.random.seed(42)


## 2. Load / Simulate RNA-Seq Count Data


In [11]:
n_genes = 500
n_samples_per_group = 25
n_de_genes = 60  # kitne genes actually differentially expressed hain (ground truth)

gene_names = [f"GENE_{i:04d}" for i in range(1, n_genes + 1)]
sample_names = [f"Normal_{i+1}" for i in range(n_samples_per_group)] + \
               [f"Tumor_{i+1}" for i in range(n_samples_per_group)]
sample_labels = np.array(["Normal"] * n_samples_per_group + ["Tumor"] * n_samples_per_group)

base_mean = np.random.uniform(20, 2000, n_genes)
dispersion = 0.3

counts = np.zeros((n_genes, len(sample_names)))
de_gene_idx = np.random.choice(n_genes, n_de_genes, replace=False)
log2fc_true = np.zeros(n_genes)
log2fc_true[de_gene_idx] = np.random.choice([-1, 1], n_de_genes) * np.random.uniform(1.5, 4, n_de_genes)

for j, label in enumerate(sample_labels):
    fold = 2 ** log2fc_true if label == "Tumor" else np.ones(n_genes)
    mean_j = base_mean * fold
    n_param = 1 / dispersion
    p_param = n_param / (n_param + mean_j)
    counts[:, j] = np.random.negative_binomial(n_param, p_param)

counts_df = pd.DataFrame(counts, index=gene_names, columns=sample_names).astype(int)
print(f"Counts matrix shape: {counts_df.shape}  (genes x samples)")
counts_df.head()


Counts matrix shape: (500, 50)  (genes x samples)


,Normal_1,Normal_2,Normal_3,Normal_4,Normal_5,Normal_6,Normal_7,Normal_8,Normal_9,Normal_10,...,Tumor_16,Tumor_17,Tumor_18,Tumor_19,Tumor_20,Tumor_21,Tumor_22,Tumor_23,Tumor_24,Tumor_25
GENE_0001,231,490,753,778,611,888,339,1251,647,711,...,85,296,799,615,921,769,1663,1336,1152,226
GENE_0002,2354,2805,696,2319,2701,1579,812,1977,1073,1809,...,1922,3582,1508,2565,1313,1836,2698,1779,2477,2570
GENE_0003,947,958,424,803,508,629,1802,4510,2141,1145,...,1714,1362,596,185,748,399,2143,870,440,2351
GENE_0004,1347,2175,1410,1742,1388,855,4349,1126,1937,1044,...,309,2604,1429,570,330,929,863,956,1145,1819
GENE_0005,338,464,328,179,107,505,332,433,558,318,...,69,163,445,95,73,392,309,163,512,210


## 3. Normalization (CPM + log2)

In [12]:
lib_sizes = counts_df.sum(axis=0)
cpm = counts_df.div(lib_sizes, axis=1) * 1e6
log_cpm = np.log2(cpm + 1)

print("Library sizes (total counts per sample):")
print(lib_sizes.describe())
log_cpm.head()


Library sizes (total counts per sample):
count        50.000000
mean     560211.640000
std       55370.626817
min      483610.000000
25%      507949.750000
50%      556868.000000
75%      612771.250000
max      642483.000000
dtype: float64


,Normal_1,Normal_2,Normal_3,Normal_4,Normal_5,Normal_6,Normal_7,Normal_8,Normal_9,Normal_10,...,Tumor_16,Tumor_17,Tumor_18,Tumor_19,Tumor_20,Tumor_21,Tumor_22,Tumor_23,Tumor_24,Tumor_25
GENE_0001,8.785578,9.938643,10.530608,10.610627,10.169726,10.838306,9.342265,11.206330,10.341908,10.478652,...,7.207342,8.910756,10.365761,9.907001,10.573515,10.291427,11.391402,11.055610,10.808999,8.471923
GENE_0002,12.131776,12.454576,10.417125,12.185675,12.313006,11.668341,10.601164,11.866337,11.071280,11.825310,...,11.696983,12.505100,11.281615,11.966158,11.084826,11.546269,12.089298,11.468588,11.913022,11.975586
GENE_0003,10.818576,10.905168,9.702779,10.656228,9.903636,10.341131,11.750701,13.055935,12.067580,11.165695,...,11.531794,11.110466,9.943250,8.177428,10.273571,9.345899,11.757125,10.437140,9.421735,11.847126
GENE_0004,11.326652,12.087665,11.435127,11.773019,11.352788,10.783701,13.021541,11.054523,11.923154,11.032530,...,9.062320,12.045157,11.204017,9.797495,9.094474,10.563924,10.445544,10.573042,10.800211,11.477111
GENE_0005,9.333673,9.860068,9.332917,8.493901,7.662052,10.024627,9.312210,9.676840,10.128585,9.319079,...,6.908737,8.052475,9.522240,7.220616,6.927238,9.320404,8.965644,8.025503,9.640079,8.366299


In [13]:
fig = px.box(
    log_cpm.melt(var_name="Sample", value_name="log2(CPM+1)"),
    x="Sample", y="log2(CPM+1)",
    color=[sample_labels[list(sample_names).index(s)] for s in log_cpm.melt(var_name="Sample")["Sample"]],
    title="Expression Distribution per Sample (Normalized)",
    labels={"color": "Condition"},
    template="plotly_white",
    color_discrete_map={"Normal": "#2E86AB", "Tumor": "#E63946"}
)
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()


## 4. Sample Clustering — PCA (Interactive)

In [14]:
pca = PCA(n_components=3)
pcs = pca.fit_transform(StandardScaler().fit_transform(log_cpm.T))

pca_df = pd.DataFrame(pcs, columns=["PC1", "PC2", "PC3"], index=sample_names)
pca_df["Condition"] = sample_labels
var_explained = pca.explained_variance_ratio_ * 100

fig = px.scatter(
    pca_df, x="PC1", y="PC2", color="Condition", hover_name=pca_df.index,
    title=f"PCA of Samples (PC1: {var_explained[0]:.1f}%, PC2: {var_explained[1]:.1f}% variance)",
    template="plotly_white", color_discrete_map={"Normal": "#2E86AB", "Tumor": "#E63946"},
    size_max=12
)
fig.update_traces(marker=dict(size=12, line=dict(width=1, color='white')))
fig.update_layout(height=550)
fig.show()

fig3d = px.scatter_3d(
    pca_df, x="PC1", y="PC2", z="PC3", color="Condition", hover_name=pca_df.index,
    title="3D PCA — Sample Clustering", template="plotly_white",
    color_discrete_map={"Normal": "#2E86AB", "Tumor": "#E63946"}
)
fig3d.update_traces(marker=dict(size=6))
fig3d.update_layout(height=600)
fig3d.show()


## 5. Differential Expression Testing

In [15]:
normal_mask = sample_labels == "Normal"
tumor_mask = sample_labels == "Tumor"

results = []
for gene in log_cpm.index:
    normal_vals = log_cpm.loc[gene, normal_mask]
    tumor_vals = log_cpm.loc[gene, tumor_mask]

    log2fc = tumor_vals.mean() - normal_vals.mean()
    tstat, pval = stats.ttest_ind(tumor_vals, normal_vals, equal_var=False)

    results.append({"gene": gene, "log2FC": log2fc, "pvalue": pval})

de_results = pd.DataFrame(results)
_, de_results["padj"], _, _ = multipletests(de_results["pvalue"], method="fdr_bh")

de_results["significant"] = (de_results["padj"] < 0.05) & (de_results["log2FC"].abs() > 1)
de_results["direction"] = np.where(
    de_results["significant"] & (de_results["log2FC"] > 0), "Up in Tumor",
    np.where(de_results["significant"] & (de_results["log2FC"] < 0), "Down in Tumor", "Not Significant")
)

de_results = de_results.sort_values("padj")
print(f"Total significant DE genes (padj<0.05, |log2FC|>1): {de_results['significant'].sum()}")
de_results.head(15)


Total significant DE genes (padj<0.05, |log2FC|>1): 60


,gene,log2FC,pvalue,padj,significant,direction
173,GENE_0174,-3.984669,2.626062e-24,1.313031e-21,True,Down in Tumor
483,GENE_0484,-4.135032,8.766970e-24,2.191742e-21,True,Down in Tumor
395,GENE_0396,-3.342863,5.179811e-22,8.633018e-20,True,Down in Tumor
208,GENE_0209,-3.019717,1.444150e-21,1.805188e-19,True,Down in Tumor
187,GENE_0188,-3.461055,2.957749e-20,2.957749e-18,True,Down in Tumor
174,GENE_0175,-3.891464,5.020257e-19,4.183548e-17,True,Down in Tumor
348,GENE_0349,3.469188,1.509583e-18,1.078273e-16,True,Up in Tumor
331,GENE_0332,-3.763039,2.186177e-18,1.334692e-16,True,Down in Tumor
486,GENE_0487,-3.551833,2.402445e-18,1.334692e-16,True,Down in Tumor
214,GENE_0215,3.162947,3.057530e-18,1.514044e-16,True,Up in Tumor


## 6. Volcano Plot (Interactive)

In [16]:
de_results["neg_log10_padj"] = -np.log10(de_results["padj"].replace(0, 1e-300))

fig = px.scatter(
    de_results, x="log2FC", y="neg_log10_padj", color="direction",
    hover_name="gene",
    hover_data={"log2FC": ":.2f", "padj": ":.2e", "neg_log10_padj": False, "direction": False},
    title="Volcano Plot — Differential Expression (Tumor vs Normal)",
    labels={"neg_log10_padj": "-log10(adjusted p-value)", "log2FC": "log2 Fold Change"},
    template="plotly_white",
    color_discrete_map={"Up in Tumor": "#E63946", "Down in Tumor": "#2E86AB", "Not Significant": "#B0B0B0"},
    opacity=0.75
)
fig.add_hline(y=-np.log10(0.05), line_dash="dash", line_color="gray", annotation_text="padj = 0.05")
fig.add_vline(x=1, line_dash="dash", line_color="gray")
fig.add_vline(x=-1, line_dash="dash", line_color="gray")
fig.update_traces(marker=dict(size=8, line=dict(width=0.5, color='white')))
fig.update_layout(height=600)
fig.show()


## 7. Heatmap of Top Differentially Expressed Genes

In [18]:
top_genes = de_results[de_results["significant"]].sort_values("padj").head(30)["gene"].tolist()
heatmap_data = log_cpm.loc[top_genes]

z = StandardScaler().fit_transform(heatmap_data.T).T  # z-score per gene across samples

order = np.argsort(sample_labels)
z_ordered = z[:, order]
samples_ordered = np.array(sample_names)[order]
labels_ordered = sample_labels[order]

fig = px.imshow(
    z_ordered,
    x=samples_ordered, y=top_genes,
    color_continuous_scale="RdBu_r", color_continuous_midpoint=0,
    aspect="auto",
    title="Top 30 DE Genes — Expression Heatmap (z-score)",
    labels=dict(color="z-score")
)
fig.update_layout(height=800, xaxis_tickangle=-45)
fig.show()


## 8. Expression of Top Individual Genes (Boxplots)

In [19]:
top4 = top_genes[:4]
fig = make_subplots(rows=2, cols=2, subplot_titles=top4)

colors = {"Normal": "#2E86AB", "Tumor": "#E63946"}
for i, gene in enumerate(top4):
    row, col = i // 2 + 1, i % 2 + 1
    for cond in ["Normal", "Tumor"]:
        vals = log_cpm.loc[gene, sample_labels == cond]
        fig.add_trace(
            go.Box(y=vals, name=cond, marker_color=colors[cond], showlegend=(i == 0)),
            row=row, col=col
        )

fig.update_layout(height=700, title_text="Top DE Genes — Expression by Condition", template="plotly_white")
fig.show()


## 9. Build a Tumor/Normal Classifier from Top DE Genes

In [ ]:
N_FEATURES = 20
classifier_genes = de_results[de_results["significant"]].sort_values("padj").head(N_FEATURES)["gene"].tolist()

X = log_cpm.loc[classifier_genes].T
y = (sample_labels == "Tumor").astype(int)  # 1 = Tumor, 0 = Normal

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

de_scaler = StandardScaler()
X_train_scaled = de_scaler.fit_transform(X_train)
X_test_scaled = de_scaler.transform(X_test)

clf = RandomForestClassifier(n_estimators=300, random_state=42)
clf.fit(X_train_scaled, y_train)

preds = clf.predict(X_test_scaled)
probs = clf.predict_proba(X_test_scaled)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, preds):.3f}")
print(f"ROC-AUC : {roc_auc_score(y_test, probs):.3f}")
print("\n", classification_report(y_test, preds, target_names=["Normal", "Tumor"]))


In [ ]:
cm = confusion_matrix(y_test, preds)
fig = px.imshow(
    cm, text_auto=True, color_continuous_scale="Blues",
    x=["Normal", "Tumor"], y=["Normal", "Tumor"],
    labels=dict(x="Predicted", y="Actual", color="Count"),
    title="Confusion Matrix — Tumor/Normal Classifier"
)
fig.update_layout(height=450, width=500)
fig.show()

importances = pd.Series(clf.feature_importances_, index=classifier_genes).sort_values(ascending=False)
fig2 = px.bar(
    importances, orientation='h',
    title="Gene Importance for Tumor/Normal Prediction",
    labels={"value": "Importance", "index": "Gene"},
    template="plotly_white", color=importances.values, color_continuous_scale="Viridis"
)
fig2.update_layout(height=550, showlegend=False, yaxis={'categoryorder': 'total ascending'})
fig2.show()


## 10.  Runtime Prediction — Apna Sample Direct Input Karein


In [22]:
N_FEATURES = 20
classifier_genes = de_results[de_results["significant"]].sort_values("padj").head(N_FEATURES)["gene"].tolist()

# Re-define and train the scaler and classifier for this cell's scope
X = log_cpm.loc[classifier_genes].T
y = (sample_labels == "Tumor").astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
de_scaler = StandardScaler()
X_train_scaled = de_scaler.fit_transform(X_train)
clf = RandomForestClassifier(n_estimators=300, random_state=42)
clf.fit(X_train_scaled, y_train)

gene_defaults = log_cpm.loc[classifier_genes].mean(axis=1).to_dict()

style_html = """
<style>
.predict-header {
    background: linear-gradient(90deg, #2E86AB, #A23B72);
    color: white; padding: 14px 20px; border-radius: 10px 10px 0 0;
    font-size: 18px; font-weight: 600; font-family: sans-serif;
}
</style>
<div class="predict-header"> RNA-Seq Sample Predictor — Tumor vs Normal</div>
"""
display(HTML(style_html))

gene_boxes = {}
box_list = []
for gene in classifier_genes:
    box = widgets.FloatText(
        value=round(gene_defaults[gene], 3),
        description=gene,
        style={'description_width': '110px'},
        layout=widgets.Layout(width='260px')
    )
    gene_boxes[gene] = box
    box_list.append(box)

grid = widgets.GridBox(box_list, layout=widgets.Layout(grid_template_columns="repeat(4, 270px)", grid_gap="8px"))

predict_btn = widgets.Button(
    description=" Sample Predict Karein",
    button_style='success',
    layout=widgets.Layout(width='250px', height='42px')
)
reset_btn = widgets.Button(
    description="↺ Reset to Average",
    button_style='',
    layout=widgets.Layout(width='200px', height='42px')
)
out = widgets.Output()

def render_result(label, proba):
    color = "#E63946" if label == "Tumor" else "#2E86AB"
    emoji = "" if label == "Tumor" else ""
    conf = proba[1]*100 if label == "Tumor" else proba[0]*100
    html = f"""
    <div style="border:2px solid {color}; border-radius:12px; padding:18px; margin-top:10px; font-family:sans-serif; background:#fafafa;">
        <div style="font-size:22px; font_weight:700; color:{color};">{emoji} Prediction: {label}</div>
        <div style="font-size:15px; margin-top:6px;">Confidence: <b>{conf:.2f}%</b></div>
        <div style="margin-top:10px; height:14px; width:100%; background:#e0e0e0; border-radius:7px; overflow:hidden;">
            <div style="height:100%; width:{proba[1]*100:.1f}%; background:linear-gradient(90deg,#2E86AB,#E63946);"></div>
        </div>
        <div style="display:flex; justify-content:space-between; font-size:12px; color:#555; margin-top:4px;">
            <span>P(Normal) = {proba[0]*100:.2f}%</span><span>P(Tumor) = {proba[1]*100:.2f}%</span>
        </div>
    </div>
    """
    display(HTML(html))

def on_predict(b):
    with out:
        clear_output()
        sample_vals = pd.DataFrame([[gene_boxes[g].value for g in classifier_genes]], columns=classifier_genes)
        sample_scaled = de_scaler.transform(sample_vals)
        pred = clf.predict(sample_scaled)[0]
        proba = clf.predict_proba(sample_scaled)[0]
        label = "Tumor" if pred == 1 else "Normal"
        render_result(label, proba)

def on_reset(b):
    for g in classifier_genes:
        gene_boxes[g].value = round(gene_defaults[g], 3)
    with out:
        clear_output()

predict_btn.on_click(on_predict)
reset_btn.on_click(on_reset)

display(grid)
display(widgets.HBox([predict_btn, reset_btn]))
display(out)

GridBox(children=(FloatText(value=7.71, description='GENE_0174', layout=Layout(width='260px'), style=Descripti…

Output()